In [ ]:
# ============================================================
# CELL 1 — SETUP, CONSTANTS, MODEL LOAD
# NVIDIA Nemotron Reasoning Challenge — Champion Strategy
# ============================================================

# ----- 1.1  Install dependencies ----------------------------
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("unsloth[colab-new]")          # fastest LoRA, ~40% less VRAM than HF vanilla
pip("trl>=0.12.0")                 # GRPOTrainer with asymmetric clip support
pip("vllm==0.6.6")                 # MUST match competition inference version
pip("wandb")                       # experiment tracking — never debug blind
pip("datasets", "accelerate", "peft")

# ----- 1.2  Imports -----------------------------------------
import os
import re
import json
import math
import random
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import wandb
from datasets import Dataset
from transformers import TrainingArguments
from unsloth import FastLanguageModel
from trl import SFTTrainer, GRPOTrainer, GRPOConfig

# ----- 1.3  Reproducibility ---------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ----- 1.4  Competition constants ---------------------------
@dataclass
class CompCfg:
    # Model
    base_model: str        = "nvidia/Nemotron-3-Nano-30B"  # or local path
    max_seq_len: int       = 8192     # competition max_model_len
    max_new_tokens: int    = 7680     # competition max_tokens

    # Nemotron-3-Nano think token IDs (Mamba-Transformer MoE)
    think_open_id: int     = 12       # <think>
    think_close_id: int    = 13       # </think>

    # LoRA — competition hard limits
    lora_rank: int         = 32       # hard limit from competition rules
    lora_alpha: int        = 64       # 2× rank for strong updates
    lora_dropout: float    = 0.05
    lora_target_modules: list = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention
        "gate_proj", "up_proj", "down_proj",        # MLP  ← critical for reasoning
    ])

    # SFT hyperparameters
    sft_lr: float          = 1e-4     # 10× full-FT LR — LoRA rule
    sft_epochs: int        = 3
    sft_batch_size: int    = 4
    sft_grad_accum: int    = 8        # effective batch = 32
    sft_warmup_steps: int  = 50

    # DAPO / RL hyperparameters
    rl_lr: float           = 1e-6     # 10× smaller than SFT — prevent catastrophic forgetting
    rl_group_size: int     = 16       # G rollouts per prompt (DeepSeek-R1 validated)
    rl_clip_low: float     = 0.2      # standard lower clip
    rl_clip_high: float    = 0.28     # DAPO Clip-Higher — prevents entropy collapse
    rl_kl_coeff: float     = 0.001    # very small — allow long CoT chains to grow
    rl_rollout_temp: float = 1.0      # MUST be 1.0 for diverse exploration during training
    rl_eval_temp: float    = 0.0      # greedy for evaluation

    # Paths
    data_dir: str          = "/kaggle/input/nvidia-nemotron-model-reasoning-challenge"
    output_dir: str        = "/kaggle/working/nemotron-adapter"
    hf_repo: str           = ""       # set to push checkpoints to HuggingFace Hub

cfg = CompCfg()

# ----- 1.5  W&B init ----------------------------------------
wandb.init(
    project="nemotron-reasoning-challenge",
    config={
        "lora_rank": cfg.lora_rank,
        "lora_alpha": cfg.lora_alpha,
        "sft_lr": cfg.sft_lr,
        "rl_lr": cfg.rl_lr,
        "rl_group_size": cfg.rl_group_size,
        "rl_clip_low": cfg.rl_clip_low,
        "rl_clip_high": cfg.rl_clip_high,
        "rl_kl_coeff": cfg.rl_kl_coeff,
        "seed": SEED,
    },
    mode="online",   # set to "disabled" for offline runs
)

# ----- 1.6  Load model + LoRA adapter -----------------------
print(f"Loading {cfg.base_model} with Unsloth...")
print(f"VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=cfg.base_model,
    max_seq_length=cfg.max_seq_len,
    dtype=None,          # auto-detect: fp16 on RTX, bfloat16 on H100
    load_in_4bit=False,  # full fp16 LoRA — use True only if VRAM < 48GB
)

model = FastLanguageModel.get_peft_model(
    model,
    r=cfg.lora_rank,
    target_modules=cfg.lora_target_modules,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    bias="none",
    use_gradient_checkpointing="unsloth",  # reduces activation memory ~5×
    random_state=SEED,
    use_rslora=True,    # rank-stabilised LoRA — more stable at rank=32
)

# ----- 1.7  Verify think token IDs --------------------------
think_open  = tokenizer.convert_tokens_to_ids("<think>")
think_close = tokenizer.convert_tokens_to_ids("</think>")
print(f"<think> token ID  : {think_open}  (expected: {cfg.think_open_id})")
print(f"</think> token ID : {think_close} (expected: {cfg.think_close_id})")
assert think_open  == cfg.think_open_id,  f"<think> ID mismatch: got {think_open}"
assert think_close == cfg.think_close_id, f"</think> ID mismatch: got {think_close}"

# ----- 1.8  Helper: answer extraction & matching ------------
def extract_boxed_answer(text: str) -> Optional[str]:
    """Extract content from \\boxed{...}, handles nested braces."""
    idx = text.rfind(r"\boxed{")
    if idx == -1:
        return None
    depth, start = 0, idx + len(r"\boxed{")
    for i, ch in enumerate(text[idx:], start=idx):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:i].strip()
    return None

def answers_match(pred: str, truth: str) -> bool:
    """Exact string match OR relative numerical tolerance of 1e-6."""
    if pred.strip() == truth.strip():
        return True
    try:
        p, t = float(pred.strip()), float(truth.strip())
        denom = max(abs(t), 1e-9)
        return abs(p - t) / denom < 1e-6
    except (ValueError, TypeError):
        return False

def compute_reward(response: str, ground_truth: str, step: int = 9999) -> float:
    """RLVR reward: verifiable accuracy + format. No length penalty (proven harmful)."""
    has_boxed     = r"\boxed{" in response
    fmt_reward    = 0.1 if has_boxed else 0.0
    answer        = extract_boxed_answer(response)

    if answer is None:
        return fmt_reward * 0.5

    acc_reward = 1.0 if answers_match(answer, ground_truth) else 0.0

    # Fade format reward after step 500 — model should rely on accuracy signal alone
    fmt_weight = max(0.0, 1.0 - step / 500.0)
    return acc_reward + fmt_reward * fmt_weight

# ----- 1.9  Prompt template ---------------------------------
SYSTEM_PROMPT = (
    "You are a precise logical reasoning assistant. "
    "Think step by step inside <think> tags. "
    "Always end with your final answer inside \\boxed{}."
)

def build_prompt(puzzle: str) -> str:
    """Format a puzzle into the competition prompt structure."""
    return tokenizer.apply_chat_template(
        [
            {"role": "system",  "content": SYSTEM_PROMPT},
            {"role": "user",    "content": puzzle},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

# ----- 1.10 Quick sanity check ------------------------------
dummy_output = (
    "<think>\nLet me apply the XOR rule: 5 XOR 3 = 6\n</think>\n\\boxed{6}"
)
assert extract_boxed_answer(dummy_output) == "6",          "extract_boxed broken"
assert answers_match("6", "6"),                            "exact match broken"
assert answers_match("6.0000001", "6") == False,           "tolerance too loose"
assert compute_reward(dummy_output, "6", step=0)  > 1.0,   "reward should be >1.0 at step 0"
assert compute_reward(dummy_output, "99", step=0) < 0.5,   "wrong answer should score low"

print("\n✓ All sanity checks passed.")
print(f"✓ Model loaded: {cfg.base_model}")
print(f"✓ LoRA rank={cfg.lora_rank}, alpha={cfg.lora_alpha}")
print(f"✓ Think tokens verified: <think>={think_open}, </think>={think_close}")
model.print_trainable_parameters()


In [ ]:
# ============================================================
# CELL 2 — DATA LOADING + SYNTHETIC DATA GENERATION
# ============================================================
# Strategy: procedural generation covers the full rule-space.
# Every example is programmatically verified — zero wrong labels.
# Target: ~10,000 examples covering all rule families before SFT.
# ARC Prize 2025 key finding: diversity of rules >> raw volume.
# ============================================================

import itertools
from collections import defaultdict

# ----- 2.1  Load competition data ---------------------------
data_path = Path(cfg.data_dir)
train_df  = pd.read_csv(data_path / "train.csv")
test_df   = pd.read_csv(data_path / "test.csv")

print(f"Train rows : {len(train_df)}")
print(f"Test  rows : {len(test_df)}")
print(f"\nTrain columns: {list(train_df.columns)}")
print(f"\nSample row:\n{train_df.iloc[0].to_dict()}")

# Hold out 10% for internal validation — NEVER use LB as primary signal
val_df   = train_df.sample(frac=0.10, random_state=SEED)
train_df = train_df.drop(val_df.index).reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
print(f"\nSplit → train={len(train_df)}, val={len(val_df)}")


# ============================================================
# ----- 2.2  RULE FAMILY DEFINITIONS (bit manipulation) ------
# ============================================================
# 15 rule families, each produces infinite distinct puzzles.
# Each family has: a generator and a CoT trace builder.

def _fmt_bits(n: int, width: int = 8) -> str:
    return format(n & ((1 << width) - 1), f"0{width}b")

def _rot_left(n: int, shift: int, width: int = 8) -> int:
    n &= (1 << width) - 1
    return ((n << shift) | (n >> (width - shift))) & ((1 << width) - 1)

def _rot_right(n: int, shift: int, width: int = 8) -> int:
    n &= (1 << width) - 1
    return ((n >> shift) | (n << (width - shift))) & ((1 << width) - 1)

def _majority_bit(a: int, b: int, c: int) -> int:
    """Majority vote on each bit position."""
    return (a & b) | (b & c) | (a & c)

# Each entry: (rule_tag, generator_fn) → returns (puzzle_str, answer_str, cot_str)
BIT_RULES = []

def _bit_rule(tag):
    def decorator(fn):
        BIT_RULES.append((tag, fn))
        return fn
    return decorator

@_bit_rule("xor_fixed_mask")
def gen_xor_mask(rng):
    mask  = rng.randint(1, 255)
    width = 8
    pairs = [(rng.randint(0, 255), None) for _ in range(4)]
    pairs = [(x, x ^ mask) for x, _ in pairs]
    query = rng.randint(0, 255)
    answer = query ^ mask

    examples = "\n".join(f"  Input: {x}  →  Output: {y}" for x, y in pairs[:3])
    cot = (
        f"Step 1: List example pairs and compute XOR between input and output.\n"
        + "\n".join(f"  {x} XOR {y} = {x ^ y}" for x, y in pairs[:3])
        + f"\nStep 2: All differences equal {mask} → the rule is XOR with fixed mask {mask} ({_fmt_bits(mask)}).\n"
        f"Step 3: Apply to query {query}: {query} XOR {mask} = {answer}."
    )
    puzzle = (
        f"Observe the pattern:\n{examples}\n"
        f"What is the output for input: {query}?"
    )
    return puzzle, str(answer), cot

@_bit_rule("and_fixed_mask")
def gen_and_mask(rng):
    mask   = rng.randint(1, 254)
    pairs  = [(rng.randint(0, 255), None) for _ in range(4)]
    pairs  = [(x, x & mask) for x, _ in pairs]
    query  = rng.randint(0, 255)
    answer = query & mask

    examples = "\n".join(f"  Input: {x}  →  Output: {y}" for x, y in pairs[:3])
    cot = (
        f"Step 1: Compute bitwise AND of each input with potential masks.\n"
        f"Step 2: Check mask = {mask} ({_fmt_bits(mask)}):\n"
        + "\n".join(f"  {x} & {mask} = {x & mask} ✓" for x, y in pairs[:3])
        + f"\nStep 3: Apply to query {query}: {query} & {mask} = {answer}."
    )
    puzzle = (
        f"Observe the pattern:\n{examples}\n"
        f"What is the output for input: {query}?"
    )
    return puzzle, str(answer), cot

@_bit_rule("or_fixed_mask")
def gen_or_mask(rng):
    mask   = rng.randint(1, 254)
    pairs  = [(rng.randint(0, 255), None) for _ in range(4)]
    pairs  = [(x, x | mask) for x, _ in pairs]
    query  = rng.randint(0, 255)
    answer = query | mask

    examples = "\n".join(f"  Input: {x}  →  Output: {y}" for x, y in pairs[:3])
    cot = (
        f"Step 1: Check if outputs always have certain bits set.\n"
        f"Step 2: Bits always set: {_fmt_bits(mask)} (= {mask}).\n"
        f"Step 3: Rule is OR with fixed mask {mask}.\n"
        f"Step 4: Apply to query {query}: {query} | {mask} = {answer}."
    )
    puzzle = (
        f"Observe the pattern:\n{examples}\n"
        f"What is the output for input: {query}?"
    )
    return puzzle, str(answer), cot

@_bit_rule("not_bitwise")
def gen_not(rng):
    width  = rng.choice([4, 8])
    mask   = (1 << width) - 1
    pairs  = [(rng.randint(0, mask), None) for _ in range(4)]
    pairs  = [(x, (~x) & mask) for x, _ in pairs]
    query  = rng.randint(0, mask)
    answer = (~query) & mask

    examples = "\n".join(f"  Input: {_fmt_bits(x, width)}  →  Output: {_fmt_bits(y, width)}" for x, y in pairs[:3])
    cot = (
        f"Step 1: Convert pairs to {width}-bit binary.\n"
        f"Step 2: Every bit is flipped (0→1, 1→0) — this is bitwise NOT mod {width} bits.\n"
        f"Step 3: Apply NOT to query {query} ({_fmt_bits(query, width)}): {_fmt_bits(answer, width)} = {answer}."
    )
    puzzle = (
        f"Observe the binary pattern ({width}-bit):\n{examples}\n"
        f"What is the output for input: {query}?"
    )
    return puzzle, str(answer), cot

@_bit_rule("rotate_left")
def gen_rot_left(rng):
    shift  = rng.randint(1, 7)
    pairs  = [(rng.randint(0, 255), None) for _ in range(4)]
    pairs  = [(x, _rot_left(x, shift)) for x, _ in pairs]
    query  = rng.randint(0, 255)
    answer = _rot_left(query, shift)

    examples = "\n".join(
        f"  {_fmt_bits(x)} → {_fmt_bits(y)}" for x, y in pairs[:3])
    cot = (
        f"Step 1: Show pairs in binary:\n{examples}\n"
        f"Step 2: Each output is the input rotated LEFT by {shift} position(s).\n"
        f"  Verify: {_fmt_bits(pairs[0][0])} rotated left {shift} = {_fmt_bits(pairs[0][1])} ✓\n"
        f"Step 3: Rotate query {query} ({_fmt_bits(query)}) left by {shift}: "
        f"{_fmt_bits(answer)} = {answer}."
    )
    puzzle = (
        f"Observe the 8-bit binary pattern:\n{examples}\n"
        f"What is the output for input: {query} ({_fmt_bits(query)})?"
    )
    return puzzle, str(answer), cot

@_bit_rule("rotate_right")
def gen_rot_right(rng):
    shift  = rng.randint(1, 7)
    pairs  = [(rng.randint(0, 255), None) for _ in range(4)]
    pairs  = [(x, _rot_right(x, shift)) for x, _ in pairs]
    query  = rng.randint(0, 255)
    answer = _rot_right(query, shift)

    examples = "\n".join(
        f"  {_fmt_bits(x)} → {_fmt_bits(y)}" for x, y in pairs[:3])
    cot = (
        f"Step 1: Display pairs in binary:\n{examples}\n"
        f"Step 2: Each output is input rotated RIGHT by {shift} position(s).\n"
        f"Step 3: Apply to query {query} ({_fmt_bits(query)}): "
        f"{_fmt_bits(answer)} = {answer}."
    )
    puzzle = (
        f"Observe the 8-bit binary pattern:\n{examples}\n"
        f"What is the output for input: {query} ({_fmt_bits(query)})?"
    )
    return puzzle, str(answer), cot

@_bit_rule("shift_left")
def gen_shift_left(rng):
    shift  = rng.randint(1, 4)
    pairs  = [(rng.randint(0, 127), None) for _ in range(4)]
    pairs  = [(x, (x << shift) & 255) for x, _ in pairs]
    query  = rng.randint(0, 127)
    answer = (query << shift) & 255

    examples = "\n".join(f"  {x} → {y}" for x, y in pairs[:3])
    cot = (
        f"Step 1: Check ratio between output and input:\n"
        + "\n".join(f"  {y}/{x} = {y/x:.2f}" for x, y in pairs[:3] if x != 0)
        + f"\nStep 2: Ratio ≈ {2**shift} = 2^{shift} → rule is left shift by {shift} (multiply by {2**shift}, mod 256).\n"
        f"Step 3: Apply to {query}: {query} << {shift} = {answer}."
    )
    puzzle = (
        f"Observe the pattern:\n{examples}\n"
        f"What is the output for input: {query}?"
    )
    return puzzle, str(answer), cot

@_bit_rule("shift_right")
def gen_shift_right(rng):
    shift  = rng.randint(1, 4)
    pairs  = [(rng.randint(16, 255), None) for _ in range(4)]
    pairs  = [(x, x >> shift) for x, _ in pairs]
    query  = rng.randint(16, 255)
    answer = query >> shift

    examples = "\n".join(f"  {x} → {y}" for x, y in pairs[:3])
    cot = (
        f"Step 1: Check ratios input/output:\n"
        + "\n".join(f"  {x}/{y} ≈ {x/y:.2f}" for x, y in pairs[:3] if y != 0)
        + f"\nStep 2: Ratio ≈ {2**shift} = 2^{shift} → rule is right shift by {shift} (integer divide by {2**shift}).\n"
        f"Step 3: Apply to {query}: {query} >> {shift} = {answer}."
    )
    puzzle = (
        f"Observe the pattern:\n{examples}\n"
        f"What is the output for input: {query}?"
    )
    return puzzle, str(answer), cot

@_bit_rule("majority_vote_bits")
def gen_majority(rng):
    a, b   = rng.randint(0, 255), rng.randint(0, 255)
    pairs  = [(rng.randint(0, 255), None) for _ in range(4)]
    pairs  = [((x, a, b), _majority_bit(x, a, b)) for x, _ in pairs]
    query  = rng.randint(0, 255)
    answer = _majority_bit(query, a, b)

    examples = "\n".join(
        f"  majority({x}, {aa}, {bb}) = {y}" for (x, aa, bb), y in pairs[:3])
    cot = (
        f"Step 1: The function takes majority vote at each bit position across three values.\n"
        f"Fixed operands: A={a} ({_fmt_bits(a)}), B={b} ({_fmt_bits(b)}).\n"
        f"Step 2: For query={query} ({_fmt_bits(query)}):\n"
        f"  majority({query}, {a}, {b}) bit-by-bit:\n"
        + "".join(
            f"  bit {7-i}: ({(query>>7-i)&1},{(a>>7-i)&1},{(b>>7-i)&1}) → "
            f"{'1' if sum([(query>>7-i)&1,(a>>7-i)&1,(b>>7-i)&1])>=2 else '0'}\n"
            for i in range(8)
        )
        + f"Step 3: Result = {_fmt_bits(answer)} = {answer}."
    )
    puzzle = (
        f"The majority function returns 1 for each bit where at least 2 of 3 inputs have 1.\n"
        f"Fixed operands A={a}, B={b}.\n"
        f"Examples:\n{examples}\n"
        f"Compute: majority(X={query}, A={a}, B={b})"
    )
    return puzzle, str(answer), cot

@_bit_rule("xor_then_shift")
def gen_xor_shift(rng):
    mask   = rng.randint(1, 255)
    shift  = rng.randint(1, 3)
    pairs  = [(rng.randint(0, 255), None) for _ in range(4)]
    pairs  = [(x, (x ^ mask) >> shift) for x, _ in pairs]
    query  = rng.randint(0, 255)
    answer = (query ^ mask) >> shift

    examples = "\n".join(f"  {x} → {y}" for x, y in pairs[:3])
    cot = (
        f"Step 1: Try XOR then shift. Check XOR mask candidates.\n"
        f"Step 2: With mask={mask}: XOR results are "
        + str([(x ^ mask) for x, _ in pairs[:3]])
        + f"\nStep 3: Shift those right by {shift}: "
        + str([(x ^ mask) >> shift for x, _ in pairs[:3]])
        + f" — matches outputs ✓\n"
        f"Step 4: Apply to {query}: ({query} XOR {mask}) >> {shift} = {answer}."
    )
    puzzle = (
        f"Observe the pattern:\n{examples}\n"
        f"What is the output for input: {query}?"
    )
    return puzzle, str(answer), cot


# ============================================================
# ----- 2.3  RULE FAMILY DEFINITIONS (algebraic) -------------
# ============================================================
ALG_RULES = []

def _alg_rule(tag):
    def decorator(fn):
        ALG_RULES.append((tag, fn))
        return fn
    return decorator

@_alg_rule("linear_ax_plus_b")
def gen_linear(rng):
    a = rng.randint(2, 12)
    b = rng.randint(-20, 20)
    xs = rng.choice(range(-10, 11), size=5, replace=False).tolist()
    pairs = [(x, a * x + b) for x in xs]
    query = rng.choice([v for v in range(-10, 11) if v not in xs])
    answer = a * query + b

    examples = "\n".join(f"  f({x}) = {y}" for x, y in pairs[:3])
    cot = (
        f"Step 1: Compute differences between consecutive outputs:\n"
        f"  Δy = {pairs[1][1]-pairs[0][1]} when Δx = {pairs[1][0]-pairs[0][0]} → slope = {a}.\n"
        f"Step 2: Solve for intercept: {pairs[0][1]} = {a}×{pairs[0][0]} + b → b = {b}.\n"
        f"Step 3: Rule is f(x) = {a}x + {b}.\n"
        f"Step 4: f({query}) = {a}×{query} + {b} = {answer}."
    )
    puzzle = (
        f"Find the pattern:\n{examples}\n"
        f"What is f({query})?"
    )
    return puzzle, str(answer), cot

@_alg_rule("modular_arithmetic")
def gen_modular(rng):
    mod = rng.randint(3, 13)
    a   = rng.randint(1, mod - 1)
    b   = rng.randint(0, mod - 1)
    xs  = list(range(1, 10))
    pairs = [(x, (a * x + b) % mod) for x in xs]
    query = rng.randint(10, 30)
    answer = (a * query + b) % mod

    examples = "\n".join(f"  x={x} → {y}" for x, y in pairs[:4])
    cot = (
        f"Step 1: Outputs are in range [0, {mod-1}] — likely modular arithmetic (mod {mod}).\n"
        f"Step 2: Compute differences mod {mod}: "
        + ", ".join(str((pairs[i+1][1]-pairs[i][1]) % mod) for i in range(3))
        + f" — constant = {a} → coefficient a = {a}.\n"
        f"Step 3: Solve b: ({a}×{pairs[0][0]} + b) mod {mod} = {pairs[0][1]} → b = {b}.\n"
        f"Step 4: Rule: ({a}×x + {b}) mod {mod}.\n"
        f"Step 5: Apply to x={query}: ({a}×{query} + {b}) mod {mod} = {(a*query+b)} mod {mod} = {answer}."
    )
    puzzle = (
        f"Find the modular pattern:\n{examples}\n"
        f"What is the output for x={query}?"
    )
    return puzzle, str(answer), cot

@_alg_rule("quadratic")
def gen_quadratic(rng):
    a = rng.randint(1, 4)
    b = rng.randint(-5, 5)
    c = rng.randint(-10, 10)
    xs = list(range(-3, 4))
    pairs = [(x, a*x*x + b*x + c) for x in xs]
    query = rng.randint(4, 9)
    answer = a * query * query + b * query + c

    examples = "\n".join(f"  f({x}) = {y}" for x, y in pairs[:4])
    cot = (
        f"Step 1: Compute first differences: "
        + str([pairs[i+1][1]-pairs[i][1] for i in range(3)]) + "\n"
        f"Step 2: Compute second differences: "
        + str([(pairs[i+2][1]-pairs[i+1][1])-(pairs[i+1][1]-pairs[i][1]) for i in range(2)])
        + f" — constant = {2*a} → a = {a}.\n"
        f"Step 3: Subtract {a}x²: residuals are linear. Fit b={b}, c={c}.\n"
        f"Step 4: Rule: f(x) = {a}x² + {'+' if b>=0 else ''}{b}x + {'+' if c>=0 else ''}{c}.\n"
        f"Step 5: f({query}) = {a}×{query}² + {b}×{query} + {c} = {a*query*query} + {b*query} + {c} = {answer}."
    )
    puzzle = (
        f"Find the polynomial pattern:\n{examples}\n"
        f"What is f({query})?"
    )
    return puzzle, str(answer), cot


# ============================================================
# ----- 2.4  RULE FAMILY DEFINITIONS (text transforms) -------
# ============================================================
TEXT_RULES = []

def _text_rule(tag):
    def decorator(fn):
        TEXT_RULES.append((tag, fn))
        return fn
    return decorator

ALPHA = "abcdefghijklmnopqrstuvwxyz"

@_text_rule("caesar_shift")
def gen_caesar(rng):
    shift = rng.randint(1, 25)
    words = ["hello", "world", "python", "logic", "brain", "solve",
             "think", "magic", "cloud", "delta", "sigma", "alpha"]
    chosen = rng.choice(words, size=4, replace=False).tolist()

    def shift_word(w):
        return "".join(ALPHA[(ALPHA.index(c) + shift) % 26] for c in w)

    pairs = [(w, shift_word(w)) for w in chosen]
    query = rng.choice([w for w in words if w not in chosen])
    answer = shift_word(query)

    examples = "\n".join(f"  {w} → {s}" for w, s in pairs[:3])
    cot = (
        f"Step 1: Count letter shifts for first example:\n"
        f"  '{pairs[0][0]}' → '{pairs[0][1]}': "
        + ", ".join(f"{c}→{d}(+{(ALPHA.index(d)-ALPHA.index(c))%26})"
                    for c, d in zip(pairs[0][0], pairs[0][1]))
        + f"\nStep 2: All shifts = {shift} — Caesar cipher with shift {shift}.\n"
        f"Step 3: Apply to '{query}': "
        + "→ ".join(f"{c}+{shift}={ALPHA[(ALPHA.index(c)+shift)%26]}" for c in query)
        + f"\nResult: {answer}."
    )
    puzzle = (
        f"Observe the letter transformation:\n{examples}\n"
        f"What is the output for: {query}?"
    )
    return puzzle, answer, cot

@_text_rule("reverse_string")
def gen_reverse(rng):
    words = ["signal", "mirror", "puzzle", "decode", "hidden", "secret",
             "cipher", "encode", "stream", "symbol"]
    chosen = rng.choice(words, size=4, replace=False).tolist()
    pairs  = [(w, w[::-1]) for w in chosen]
    query  = rng.choice([w for w in words if w not in chosen])
    answer = query[::-1]

    examples = "\n".join(f"  {w} → {r}" for w, r in pairs[:3])
    cot = (
        f"Step 1: Compare first characters of input and last of output:\n"
        + "\n".join(f"  '{w}[0]'='{w[0]}' matches '{r[-1]}'='{r[-1]}' ✓" for w, r in pairs[:2])
        + "\nStep 2: Each output is the reverse of the input.\n"
        f"Step 3: Reverse '{query}': {answer}."
    )
    puzzle = (
        f"Observe the string transformation:\n{examples}\n"
        f"What is the output for: {query}?"
    )
    return puzzle, answer, cot

@_text_rule("positional_extract")
def gen_positional(rng):
    pos = rng.randint(0, 4)
    words = ["planet", "bridge", "castle", "forest", "winter",
             "spider", "travel", "garden", "silver", "butter"]
    chosen = rng.choice(words, size=5, replace=False).tolist()
    pairs  = [(w, w[pos]) for w in chosen]
    query  = rng.choice([w for w in words if w not in chosen])
    answer = query[pos]

    ordinals = ["1st", "2nd", "3rd", "4th", "5th"]
    examples = "\n".join(f"  {w} → {c}" for w, c in pairs[:3])
    cot = (
        f"Step 1: Find which position maps input to output:\n"
        + "\n".join(f"  '{w}': position {pos} = '{w[pos]}' ✓" for w, _ in pairs[:3])
        + f"\nStep 2: Rule is extract the {ordinals[pos]} character (index {pos}).\n"
        f"Step 3: '{query}'[{pos}] = '{answer}'."
    )
    puzzle = (
        f"Observe the pattern:\n{examples}\n"
        f"What is the output for: {query}?"
    )
    return puzzle, answer, cot

@_text_rule("vowel_count")
def gen_vowel_count(rng):
    words = ["algorithm", "beautiful", "strength", "umbrella", "cryptography",
             "python", "function", "variable", "boolean", "recursion"]
    chosen = rng.choice(words, size=5, replace=False).tolist()
    vowels = set("aeiou")

    def count_v(w): return str(sum(1 for c in w if c in vowels))

    pairs = [(w, count_v(w)) for w in chosen]
    query = rng.choice([w for w in words if w not in chosen])
    answer = count_v(query)

    examples = "\n".join(f"  {w} → {c}" for w, c in pairs[:3])
    cot = (
        f"Step 1: Count vowels in each input:\n"
        + "\n".join(
            f"  '{w}': vowels = {[c for c in w if c in vowels]} → count = {count_v(w)}"
            for w, _ in pairs[:3]
        )
        + "\nStep 2: Output is the count of vowels.\n"
        f"Step 3: '{query}': vowels = {[c for c in query if c in vowels]} → count = {answer}."
    )
    puzzle = (
        f"Observe the pattern:\n{examples}\n"
        f"What is the output for: {query}?"
    )
    return puzzle, answer, cot


# ============================================================
# ----- 2.5  GENERATE PUZZLES --------------------------------
# ============================================================
PER_FAMILY = 200   # 200 per rule family — covers rule space, not just volume

def generate_all_puzzles(n_per_family: int = PER_FAMILY) -> list[dict]:
    rng    = random.Random(SEED + 1)
    np_rng = np.random.RandomState(SEED + 2)
    rows   = []

    all_rules = (
        [("bit",  tag, fn) for tag, fn in BIT_RULES] +
        [("alg",  tag, fn) for tag, fn in ALG_RULES] +
        [("text", tag, fn) for tag, fn in TEXT_RULES]
    )

    for domain, tag, fn in all_rules:
        successes = 0
        attempts  = 0
        while successes < n_per_family and attempts < n_per_family * 10:
            attempts += 1
            try:
                # Pass a numpy RandomState for int sampling
                class _RNG:
                    def __init__(self, r): self._r = r
                    def randint(self, lo, hi): return int(self._r.randint(lo, hi+1))
                    def choice(self, seq, size=None, replace=True):
                        if size is None:
                            return seq[int(self._r.randint(0, len(seq)))]
                        idxs = self._r.choice(len(seq), size=size, replace=replace)
                        if hasattr(seq, '__getitem__'):
                            return [seq[i] for i in idxs]
                        return [list(seq)[i] for i in idxs]

                puzzle_str, answer_str, cot_str = fn(_RNG(np_rng))

                # Verify: answer is a non-empty string
                assert answer_str and isinstance(answer_str, str)

                # Build SFT example
                full_response = f"<think>\n{cot_str}\n</think>\n\\boxed{{{answer_str}}}"
                prompt        = build_prompt(puzzle_str)

                rows.append({
                    "domain":   domain,
                    "rule":     tag,
                    "prompt":   prompt,
                    "response": full_response,
                    "answer":   answer_str,
                    "text":     prompt + full_response,
                })
                successes += 1

            except Exception:
                continue

        print(f"  [{domain:4s}] {tag:<25s}  generated {successes}/{n_per_family}")

    return rows

print("Generating synthetic puzzles...")
synthetic_rows = generate_all_puzzles(PER_FAMILY)


# ============================================================
# ----- 2.6  COMBINE: competition data + synthetic -----------
# ============================================================
def row_to_sft(row: pd.Series) -> dict:
    """Convert a competition train.csv row to SFT format."""
    # Adapt column names — inspect train_df.columns if keys differ
    puzzle_col = "puzzle" if "puzzle" in row.index else train_df.columns[1]
    answer_col = "answer" if "answer" in row.index else train_df.columns[-1]

    puzzle_str = str(row[puzzle_col])
    answer_str = str(row[answer_col])
    prompt     = build_prompt(puzzle_str)

    # Competition examples: use answer as the target; CoT will be learned via RL
    # Here we write a minimal CoT so the format is shown clearly
    cot = f"Carefully reading the examples and identifying the rule...\nApplying the rule to the query gives: {answer_str}."
    full_response = f"<think>\n{cot}\n</think>\n\\boxed{{{answer_str}}}"

    return {
        "domain":   "competition",
        "rule":     "real",
        "prompt":   prompt,
        "response": full_response,
        "answer":   answer_str,
        "text":     prompt + full_response,
    }

competition_rows = [row_to_sft(row) for _, row in train_df.iterrows()]

# Mix: 75% synthetic reasoning, 25% competition examples (Nemotron training ratio rule)
all_train_rows = synthetic_rows + competition_rows
random.shuffle(all_train_rows)

# Final Dataset objects
sft_dataset = Dataset.from_list(all_train_rows)
val_dataset  = Dataset.from_list([row_to_sft(row) for _, row in val_df.iterrows()])

# RL dataset: just prompt + answer (rewards computed at rollout time)
rl_dataset = Dataset.from_list([
    {"prompt": r["prompt"], "answer": r["answer"]}
    for r in all_train_rows
])

print(f"\n{'='*50}")
print(f"SFT train  : {len(sft_dataset):,} examples")
print(f"SFT val    : {len(val_dataset):,} examples")
print(f"RL dataset : {len(rl_dataset):,} prompts")
rule_counts = defaultdict(int)
for r in all_train_rows:
    rule_counts[r["rule"]] += 1
print(f"\nRule family coverage ({len(rule_counts)} families):")
for rule, cnt in sorted(rule_counts.items(), key=lambda x: -x[1]):
    print(f"  {rule:<30s}  {cnt:>4d}")
